# 02c | Group By and Aggregation

"Group by one column, then compute statistics for each group" is the single most common operation in data analysis interviews. It corresponds to dplyr's `group_by() %>% summarise()` and SQL's `GROUP BY`.

**After this lesson you can:** compute all kinds of statistics by group, calculate within-group shares, and take the top few of each group. About 50 minutes.

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()
employees

## 1. The basic syntax

```python
employees.groupby("dept")["salary"].mean()
```

Read it as: **group by dept → take the salary column → compute the mean for each group**.

The result is a Series, with the grouping column (dept) moved to the index. Add `.reset_index()` to turn it back into a regular column and get a normal table.

In [ ]:
print(employees.groupby("dept")["salary"].mean())
employees.groupby("dept")["salary"].mean().reset_index()

### ✏️ Exercise 1 (Fill in the blanks)

Compute each user's total order amount as a table with two columns, `user_id` and `amount`, and store it in `out`.

In [ ]:
out = orders.groupby(____)[____].____().reset_index()
out

<details>
<summary><b>Answer</b></summary>

```python
out = orders.groupby("user_id")["amount"].sum().reset_index()
out
```

</details>

## 2. Several statistics at once: named aggregation

The syntax is `new_col=("source_col", "function")`, which maps one-to-one to dplyr's `summarise(n = n(), total = sum(amount))`.

Common function names: `"count"`, `"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`, `"std"`, `"nunique"`.

In [ ]:
summary = (
    employees.groupby("dept")
    .agg(n=("emp_id", "count"), avg_salary=("salary", "mean"), max_salary=("salary", "max"))
    .reset_index()
)
summary

### ✏️ Exercise 2 (Write it yourself)

Looking only at orders with status `completed`, compute for each user:

- `n_orders`: the number of orders;
- `total`: the total amount;
- `avg`: the average amount.

The result has four columns, `user_id`, `n_orders`, `total`, `avg`; store it in `out`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

First filter with `orders[orders["status"] == "completed"]`, then `.groupby("user_id").agg(n_orders=("order_id", "count"), ...)`, and finally `.reset_index()`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = (
    orders[orders["status"] == "completed"]
    .groupby("user_id")
    .agg(n_orders=("order_id", "count"), total=("amount", "sum"), avg=("amount", "mean"))
    .reset_index()
)
out
```

</details>

## 3. Grouping by several columns, and pivot tables

`groupby(["a", "b"])` groups by several columns. If you want to turn one of them "sideways" into column headers, use `pivot_table` (R's `pivot_wider` / `dcast`).

In [ ]:
m = users.merge(orders, on="user_id")      # merging is covered in the next lesson; just use it for now
print(m.groupby(["country", "status"])["amount"].sum().reset_index())

m.pivot_table(index="country", columns="status", values="amount", aggfunc="sum", fill_value=0)

## 4. transform: compute, then attach back to every row

`agg` collapses each group into one row. Sometimes you want to keep every row and just add a column holding "the statistic for that row's group," for example "each person's salary as a share of the department's total salary." That's what `transform` is for; it's dplyr's `group_by() %>% mutate()`.

In [ ]:
e = employees.copy()
e["dept_avg"] = e.groupby("dept")["salary"].transform("mean")
e["diff_from_avg"] = e["salary"] - e["dept_avg"]
e[["name", "dept", "salary", "dept_avg", "diff_from_avg"]]

### ✏️ Exercise 3 (Write it yourself)

Add a column `share` to `e`, a copy of `employees`: each person's salary as a share of **the total salary of their department**.

In [ ]:
e = employees.copy()
# Write your code below

<details>
<summary><b>Hint</b></summary>

The department total is `e.groupby("dept")["salary"].transform("sum")`; it has the same length as `e`, so you can divide directly.

</details>

<details>
<summary><b>Answer</b></summary>

```python
e = employees.copy()
e["share"] = e["salary"] / e.groupby("dept")["salary"].transform("sum")
e[["name", "dept", "salary", "share"]]
```

</details>

## 5. Within-group ranking and the top N per group

Questions like "the 2 highest-paid people in each department" are called **top-N per group**, and they come up in every interview. Two ways to write it:

**Approach 1: sort first, then take the first N rows of each group**

```python
df.sort_values("salary", ascending=False).groupby("dept").head(2)
```

**Approach 2: compute the within-group rank first, then filter**

```python
df["rk"] = df.groupby("dept")["salary"].rank(method="dense", ascending=False)
df[df["rk"] <= 2]
```

The `method` argument of `rank` decides how ties are handled; the options map to SQL's three ranking functions (you'll see them again in 03e):

| Salary | `"first"` (ROW_NUMBER) | `"min"` (RANK) | `"dense"` (DENSE_RANK) |
|---|---|---|---|
| 150 | 1 | 1 | 1 |
| 120 | 2 | 2 | 2 |
| 120 | 3 | 2 | 2 |
| 95 | 4 | 4 | 3 |

In [ ]:
e = employees.copy()
e["rk"] = e.groupby("dept")["salary"].rank(method="dense", ascending=False)
e.sort_values(["dept", "rk"])[["dept", "name", "salary", "rk"]]

### ✏️ Exercise 4 (Write it yourself)

Find **the highest-paid person in each department**, with the three columns `dept`, `name`, `salary`, and store it in `out`. (In this data there are no ties for the top salary within a department.)

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

Sort by salary descending, then `.groupby("dept").head(1)`, and finally select the three columns.

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = (
    employees.sort_values("salary", ascending=False)
    .groupby("dept")
    .head(1)
    [["dept", "name", "salary"]]
)
out
```

</details>

## 6. Within-group running totals and "the previous row"

- `groupby(...)[col].cumsum()`: within-group cumulative sum;
- `groupby(...)[col].shift(1)`: the previous row's value within the group (SQL's `LAG`);
- `groupby(...).cumcount()`: the row number within the group (starting at 0).

**Always sort by time first** before using these.

In [ ]:
o = orders.sort_values(["user_id", "order_date"]).copy()
o["prev_amount"] = o.groupby("user_id")["amount"].shift(1)
o["nth_order"] = o.groupby("user_id").cumcount() + 1
o[["user_id", "order_date", "amount", "prev_amount", "nth_order"]].head(6)

### ✏️ Exercise 5 (Write it yourself)

For `orders`, by user and in date order, compute each user's **cumulative spending** in a new column `running_total`. The result table `o` must contain `order_id` and `running_total`.

In [ ]:
o = orders.sort_values(["user_id", "order_date"]).copy()
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
o = orders.sort_values(["user_id", "order_date"]).copy()
o["running_total"] = o.groupby("user_id")["amount"].cumsum()
o[["user_id", "order_date", "amount", "running_total"]]
```

</details>

## Summary

- `df.groupby("g")["x"].mean().reset_index()`;
- Several statistics: `.agg(new_name=("col", "function"))`;
- Keep every row: `.transform("mean")`;
- Top N per group: sort, then `.groupby("g").head(N)`, or use `.rank()`;
- Running total `cumsum()`, previous row `shift(1)`; sort first.

Move on to **02d**.